# 09 · Model comparison

Compares all models on the same 2,737 test texts, using the saved predictions from steps 05–07. No training and no GPU needed; runs in a few minutes.

In [ ]:
%pip install -q datasets pandas scipy matplotlib python-dotenv

## Settings and data

In [ ]:
import difflib
import json
import os
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datasets import load_dataset
from dotenv import load_dotenv
from scipy.stats import fisher_exact

REPO = Path.cwd().parent
load_dotenv(REPO / ".env")
HF_DATASET_REPO = "aliissa824/german-academic-ai-text"
RESULTS = Path("results")
RESULTS.mkdir(exist_ok=True)

LABELS = ["human", "ai_generated", "ai_assisted"]
PREDICTIONS = {   # model → saved test predictions
    "TF-IDF": REPO / "06_tfidf-baseline/results/tfidf-test-predictions.csv",
    "DistilBERT German": REPO / "07_fine-tuning/results/distilbert-german/test-predictions.csv",
    "XLM-RoBERTa base": REPO / "07_fine-tuning/results/xlm-roberta-base/test-predictions.csv",
    "mmBERT-small": REPO / "07_fine-tuning/results/mmbert-small/test-predictions.csv",
}
TRAINED = list(PREDICTIONS)

test = load_dataset(HF_DATASET_REPO, split="test", token=os.getenv("HF_TOKEN")).to_pandas()
for name, path in PREDICTIONS.items():
    test[name] = test["id"].map(pd.read_csv(path).set_index("id")["predicted_label"])

# Zero-shot LLM (step 05): last valid answer per text; 6 texts were blocked by the content filter
answers = {}
for line in open(REPO / "05_zero-shot-llm/results/luna6-test-predictions.jsonl", encoding="utf-8"):
    row = json.loads(line)
    if pd.notna(row.get("predicted_label")) and not row.get("error"):
        answers[row["id"]] = int(row["predicted_label"])
test["GPT-6 Luna zero-shot"] = test["id"].map(answers)

assert test[TRAINED].notna().all().all(), "a prediction file doesn't cover the whole test set"
print(f"{len(test):,} test texts, {test['GPT-6 Luna zero-shot'].notna().sum():,} with a zero-shot answer")

## 1 · Main table

Accuracy with a 95% range. The range comes from resampling whole documents (2,000 times), because texts from the same document, or a human text and its AI partners, aren't independent.

In [ ]:
rng = np.random.default_rng(0)
doc_rows = test.groupby("document_id").indices
docs = np.array(list(doc_rows))
samples = [np.concatenate([doc_rows[d] for d in rng.choice(docs, len(docs))]) for _ in range(2000)]
correct = {m: (test[m] == test["label"]).to_numpy() for m in TRAINED}


def class_errors(pred, label):
    return {f"{name} error": float((pred[label == i] != i).mean()) for i, name in enumerate(LABELS)}


rows = []
for model in ["GPT-6 Luna zero-shot", *TRAINED]:
    part = test.dropna(subset=[model])
    pred, label = part[model].astype(int).to_numpy(), part["label"].to_numpy()
    row = {"model": model, "texts": len(part), "accuracy": float((pred == label).mean()), **class_errors(pred, label)}
    if model in TRAINED:
        boot = [correct[model][s].mean() for s in samples]
        row["accuracy low"], row["accuracy high"] = np.percentile(boot, [2.5, 97.5])
    rows.append(row)

# Hassan's two models: only summary numbers exist (from step 07)
summary = pd.read_csv(REPO / "07_fine-tuning/results/all-models.csv").set_index("model")
for key, name in [("mmbert-base", "mmBERT-base"), ("gelectra-large", "GELECTRA-large")]:
    s = summary.loc[key]
    rows.append({"model": name, "texts": len(test), "accuracy": s["test_accuracy"], "human error": s["test_error_human"],
                 "ai_generated error": s["test_error_ai_generated"], "ai_assisted error": s["test_error_ai_assisted"]})

comparison = pd.DataFrame(rows)
comparison.round(4).to_csv(RESULTS / "comparison.csv", index=False)
print(comparison.set_index("model").map(lambda v: f"{v:.1%}" if isinstance(v, float) and v <= 1 else v).to_string())

## 2 · Which differences are real?

Difference in accuracy between two models on the same texts, with a 95% range (same resampling as above). If the range doesn't include 0, the difference is real.

In [ ]:
pairs = [("mmBERT-small", "XLM-RoBERTa base"), ("mmBERT-small", "TF-IDF"), ("XLM-RoBERTa base", "DistilBERT German"),
         ("XLM-RoBERTa base", "TF-IDF"), ("DistilBERT German", "TF-IDF")]
diffs = []
for a, b in pairs:
    boot = [correct[a][s].mean() - correct[b][s].mean() for s in samples]
    low, high = np.percentile(boot, [2.5, 97.5])
    diffs.append({"model": a, "vs": b, "difference": correct[a].mean() - correct[b].mean(), "low": low, "high": high,
                  "real": "yes" if low > 0 or high < 0 else "no"})
differences = pd.DataFrame(diffs)
differences.round(4).to_csv(RESULTS / "differences.csv", index=False)
print(differences.to_string(index=False, float_format=lambda v: f"{v:+.2%}"))

## 3 · Is DeepSeek harder, or its source?

DeepSeek mostly worked on peDOCS texts. Comparing within peDOCS separates the two.

In [ ]:
assisted = test[test["label"] == 2]
print(pd.crosstab(assisted["generator_model"], assisted["source"], margins=True), "\n")

by_cell = []
for model in TRAINED:
    error = assisted[model] != 2
    table = (error.groupby([assisted["generator_model"], assisted["source"]]).agg(["size", "mean"])
             .rename(columns={"size": "texts", "mean": "error"}).reset_index())
    by_cell.append(table.assign(model=model))
by_cell = pd.concat(by_cell)
by_cell.round(4).to_csv(RESULTS / "ai-assisted-error-by-generator-and-source.csv", index=False)

pedocs = assisted[assisted["source"] == "pedocs"]
for model in TRAINED:
    error = pedocs[model] != 2
    table = [[int(error[pedocs["generator_model"] == g].sum()), int((~error)[pedocs["generator_model"] == g].sum())]
             for g in ["deepseek-flash", "gpt-6-luna"]]
    print(f"{model:18} peDOCS AI-assisted error: DeepSeek {table[0][0] / sum(table[0]):.1%}, "
          f"GPT-6 Luna {table[1][0] / sum(table[1]):.1%}  (Fisher p = {fisher_exact(table)[1]:.4f})")

## 4 · How much does each AI model change when it polishes?

Word-level similarity between each AI-assisted text and its human original (1.0 = unchanged).

In [ ]:
texts = test.set_index("id")["text"]
words = lambda s: re.findall(r"\w+", s.lower())

sim_rows = []
for _, row in assisted.iterrows():
    original = f"human-{int(row['seed_sample_id'])}"   # stored as float (empty for human rows)
    if original in texts.index:
        similarity = difflib.SequenceMatcher(None, words(texts[original]), words(row["text"]), autojunk=False).ratio()
        sim_rows.append({"id": row["id"], "generator_model": row["generator_model"], "similarity": similarity,
                         "missed": row["mmBERT-small"] != 2})
similarity = pd.DataFrame(sim_rows)
similarity.round(4).to_csv(RESULTS / "rewrite-similarity.csv", index=False)

print(similarity.groupby("generator_model")["similarity"].median().round(3).to_string(), "\n")
bins = pd.cut(similarity["similarity"], [0, 0.6, 0.7, 0.8, 0.9, 1.0], labels=["≤60%", "60–70%", "70–80%", "80–90%", ">90%"])
miss_by_similarity = similarity.groupby(bins, observed=True)["missed"].agg(texts="size", missed="mean")
print(miss_by_similarity.to_string(float_format=lambda v: f"{v:.1%}"))

## 5 · Texts that fool every trained model

In [ ]:
all_wrong = test[np.all([test[m] != test["label"] for m in TRAINED], axis=0)]
all_wrong[["id", "label", "source", "generator_model", "word_count", "mmBERT-small"]].to_csv(
    RESULTS / "hard-texts.csv", index=False)   # ids only, no texts
print(f"{len(all_wrong)} texts wrong in all {len(TRAINED)} models")
print("by class:", all_wrong["label"].map(dict(enumerate(LABELS))).value_counts().to_dict())
print(f"≤70 words: {(all_wrong['word_count'] <= 70).mean():.0%} (whole test set: {(test['word_count'] <= 70).mean():.0%})")

humans = test[test["label"] == 0]
print("\nHuman texts wrongly flagged, by source:")
print(humans[TRAINED].ne(0).groupby(humans["source"]).mean().map("{:.1%}".format).to_string())

## Charts

In [ ]:
INK, MUTED, GRID, SURFACE, BLUE = "#0b0b0b", "#52514e", "#e6e5e0", "#fcfcfb", "#2a78d6"
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 11})

# Accuracy with 95% ranges (the zero-shot LLM, at 51.7%, is left out so the differences stay visible)
chart = comparison[comparison["model"] != "GPT-6 Luna zero-shot"].sort_values("accuracy")
fig, ax = plt.subplots(figsize=(8.6, 4.2), dpi=160)
fig.patch.set_facecolor(SURFACE); ax.set_facecolor(SURFACE)
for y, (_, r) in enumerate(chart.iterrows()):
    has_range = pd.notna(r.get("accuracy low"))
    if has_range:
        ax.plot([r["accuracy low"] * 100, r["accuracy high"] * 100], [y, y], color=BLUE, lw=2, solid_capstyle="round")
    ax.scatter(r["accuracy"] * 100, y, s=70, color=BLUE if has_range else SURFACE, edgecolor=BLUE, lw=2, zorder=3)
    ax.text(r["accuracy"] * 100, y + 0.28, f"{r['accuracy']:.1%}", ha="center", fontsize=9.5, color=INK)
ax.set_yticks(range(len(chart)), [m + ("" if pd.notna(r) else " *") for m, r in zip(chart["model"], chart["accuracy low"])])
ax.set_xlim(89, 98.5)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0f}%"))
ax.grid(axis="x", color=GRID, lw=0.8); ax.set_axisbelow(True)
for side in ["top", "right", "left"]: ax.spines[side].set_visible(False)
ax.spines["bottom"].set_color("#b8b7b1"); ax.tick_params(colors=MUTED, length=0)
ax.set_title("Test accuracy with 95% range", loc="left", fontsize=13, color=INK, pad=10)
fig.text(0.01, 0.01, "* summary numbers only, no range", fontsize=9, color=MUTED)
fig.tight_layout(rect=(0, 0.04, 1, 1)); fig.savefig(RESULTS / "comparison.png", facecolor=SURFACE); plt.show()

# How often mmBERT-small misses an AI-assisted text, by how much of the original wording was kept
fig, ax = plt.subplots(figsize=(7.6, 3.8), dpi=160)
fig.patch.set_facecolor(SURFACE); ax.set_facecolor(SURFACE)
x = np.arange(len(miss_by_similarity))
ax.bar(x, miss_by_similarity["missed"] * 100, width=0.62, color=BLUE)
for i, (n, m) in enumerate(zip(miss_by_similarity["texts"], miss_by_similarity["missed"])):
    ax.text(i, m * 100 + 0.6, f"{m:.0%}", ha="center", fontsize=10, color=INK)
ax.set_xticks(x, [f"{b}\n{n} texts" for b, n in zip(miss_by_similarity.index, miss_by_similarity["texts"])]); ax.set_ylim(0, max(miss_by_similarity["missed"]) * 100 + 5)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0f}%"))
ax.set_xlabel("How similar the rewrite is to the human original (word level)", color=MUTED, labelpad=8)
ax.grid(axis="y", color=GRID, lw=0.8); ax.set_axisbelow(True)
for side in ["top", "right", "left"]: ax.spines[side].set_visible(False)
ax.spines["bottom"].set_color("#b8b7b1"); ax.tick_params(colors=MUTED, length=0)
ax.set_title("AI-assisted texts missed by mmBERT-small", loc="left", fontsize=13, color=INK, pad=10)
fig.tight_layout(); fig.savefig(RESULTS / "missed-by-similarity.png", facecolor=SURFACE); plt.show()